In [1]:
import pandas as pd
import numpy as np
import json
from scipy.spatial.distance import cdist
run_id="dwja0cgbo3"
rel_name="final19"
rel_path="../../relations_4loop"
embs_path="../../embs"
max_to_show = 25

In [2]:
def normalize(mydf):
    indices=mydf.index
    a=mydf.to_numpy()
    row_sums = a.sum(axis=1)
    new_matrix = a / row_sums[:, np.newaxis]
    mydf=pd.DataFrame(new_matrix, index=indices)
    return mydf

In [3]:
def get_nearest_points_in_df(point_df, points_df,k):
    inds = np.argpartition(cdist([point_df], points_df,metric='euclidean'),k)[0][:k]
    #inds = [cdist([point_df], points_df,metric='cosine').argmin()]
    return [points_df.iloc[ind].name for ind in inds]

In [28]:
def evaluate_relation_embs(rel_name,run_id,rel_path,embs_path,max_to_show,mode=2):
    #open the json with the relation in it
    with open(f'{rel_path}/rel_instances_{rel_name}.json') as json_data:
        rel_dicts = json.load(json_data)

    metadata = pd.read_csv(f'{embs_path}/{run_id}_{rel_name}_metadata.tsv', sep='\t')
    encodings = pd.read_csv(f'{embs_path}/{run_id}_{rel_name}_encodings.tsv', sep='\t', header=None)

    fulldf_metadata = pd.read_csv(f'{embs_path}/{run_id}_valid_metadata.tsv', sep='\t')
    fulldf_encodings = pd.read_csv(f'{embs_path}/{run_id}_valid_encodings.tsv', sep='\t', header=None)
    
    df = metadata.join(encodings, how='outer')
    df=df.drop_duplicates()
    
    fulldf = fulldf_metadata.join(fulldf_encodings, how='outer')
    fulldf=fulldf.drop_duplicates()
    mega_df=pd.concat([df,fulldf])   
    mega_df=mega_df.drop_duplicates()
    mega_df=mega_df.set_index(["word"])
    mega_df=mega_df.drop(columns=["coef"])
    mega_df= mega_df[~mega_df.index.duplicated(keep='first')]

    df=df.set_index(["word"])
    df=df.drop(columns=["coef"])
    df = df[~df.index.duplicated(keep='first')]
    #normalize everything in the df
    
    fulldf_withcoeffs=fulldf.set_index(["word","coef"])
    fulldf=fulldf.set_index(["word"])
    fulldf=fulldf.drop(columns=["coef"])
    fulldf = fulldf[~fulldf.index.duplicated(keep='first')]
    
    if mode==0:
    #does one rel make a circuit
        for ind,instance in enumerate(rel_dicts):
            print(instance)
            vecsum = np.zeros(256)
            mywords=[]
            heldcoeff=1
            for word_ind,(word,coeff) in enumerate(instance.items()):
                #get the word vec
                myvec=df.loc[word].to_numpy()*coeff[1]
                vecsum=vecsum + myvec
            eps=np.linalg.norm(vecsum)    
            print(f"eps={eps}")
            print("")
        
    if mode==1:    
        #find the heldout vector
        for ind,instance in enumerate(rel_dicts):
            print(instance)
            vecsum = np.zeros(256)
            mywords=[]
            heldcoeff=1
            for word_ind,(word,coeff) in enumerate(instance.items()):
                #get the word vec
                myvec=df.loc[word].to_numpy()*coeff[1]
                if word_ind == 0:
                    print(f"word={word},coef={coeff[0]}")
                    heldcoeff=coeff[1]
                    continue
                mywords+=[word]   
                vecsum=vecsum + myvec   
            vecsum=vecsum/((-1*heldcoeff))   
            nearests=get_nearest_points_in_df(pd.Series(vecsum),mega_df,len(instance.keys())+5)
            mynearests = [item for item in nearests if item not in mywords]
            print(f"nearest={mynearests}")
            print("")

            if ind > max_to_show:
                return

    #pairs of rels, K-M+W-Q=0 style (finding parallelograms)
    if mode==2:
        top1_acc=0
        top5_acc=0
        counter=0
        #find the heldout vector
        for ind,instance in enumerate(rel_dicts):
            for ind2,instance2 in enumerate(rel_dicts):
                #print(instance2)
                vecsum = np.zeros(256)
                mywords=[]
                heldword=""
                for word_ind,(word,coeff) in enumerate(instance.items()):
                    #get the word vec
                    myvec=df.loc[word].to_numpy()*coeff[1]
                    mywords+=[word]   
                    vecsum=vecsum + myvec   
                
                for word_ind2,(word2,coeff2) in enumerate(instance2.items()):
                    #get the word vec
                    if word_ind2 == 0:
                        #print(f"word={word2},coef={coeff2[0]}")
                        heldword=word2
                        heldcoeff=coeff2[1]
                        continue
                    myvec=df.loc[word2].to_numpy()*coeff2[1]
                    mywords+=[word2]
                    vecsum=vecsum - myvec                   
                vecsum=vecsum/(heldcoeff)   
                nearests=get_nearest_points_in_df(pd.Series(vecsum),mega_df,len(instance.keys())+5)
                mynearests = [item for item in nearests if item not in mywords]
                if mynearests[0] == heldword:
                    top1_acc += 1
                if heldword in mynearests[:5]:
                    top5_acc += 1
                counter += 1
                if counter > 1000: break    
            break       
        print(top1_acc,top5_acc,counter)            
        top1_acc_percent=top1_acc/(counter)
        top5_acc_percent=top5_acc/(counter)
        print(top1_acc_percent)
        print(top5_acc_percent)
                #print(f"nearest={mynearests}")
                #print("")

                #if ind*ind2 > max_to_show:
                #    return    
        
        

In [29]:
evaluate_relation_embs(rel_name,run_id,rel_path,embs_path,max_to_show,2)

410 973 1001
0.4095904095904096
0.972027972027972
